# 7.4 Vision Transformer

這份 Notebook 使用小型合成影像資料建立簡化版 Vision Transformer，示範 patch、patch embedding、position embedding 與 Transformer encoder 分類流程。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立方向圖形影像資料


In [ ]:
CLASS_NAMES = np.array(['vertical', 'horizontal', 'diagonal'])
IMAGE_SIZE = 32
PATCH_SIZE = 4
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2

def draw_shape(label, rng, image_size=IMAGE_SIZE, noise=0.12, jitter=4, thickness=2):
    image = rng.normal(loc=0.08, scale=noise, size=(image_size, image_size, 3)).astype('float32')
    color = np.array([0.9, 0.92, 0.88], dtype='float32') + rng.normal(0, 0.03, size=3).astype('float32')
    center = image_size // 2 + int(rng.integers(-jitter, jitter + 1))
    shift = int(rng.integers(-jitter, jitter + 1))
    if label == 0:
        image[:, max(0, center-thickness):min(image_size, center+thickness+1), :] = color
    elif label == 1:
        image[max(0, center-thickness):min(image_size, center+thickness+1), :, :] = color
    else:
        for row in range(image_size):
            col = row + shift
            if 0 <= col < image_size:
                image[row, max(0, col-thickness):min(image_size, col+thickness+1), :] = color
    image += rng.normal(0, noise * 0.5, size=image.shape).astype('float32')
    return np.clip(image, 0.0, 1.0).astype('float32')

def make_image_dataset(samples_per_class=220, seed=42):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            X.append(draw_shape(label, rng))
            y.append(label)
    X = np.stack(X).astype('float32')
    y = np.array(y, dtype='int64')
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

X, y = make_image_dataset(samples_per_class=220, seed=12)
x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
x_valid, x_test, y_valid, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(x_train.shape, x_valid.shape, x_test.shape)


## 3. 觀察圖片與 patches


In [ ]:
plt.figure(figsize=(7, 3))
for i in range(6):
    ax = plt.subplot(2, 3, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()


## 4. 定義 patch layer 與 Transformer block


In [ ]:
class Patches(tf.keras.layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        patch_dims = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dims])

class PatchEncoder(tf.keras.layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = tf.keras.layers.Dense(units=projection_dim)
        self.position_embedding = tf.keras.layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

class PositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.position_embedding = tf.keras.layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        return inputs + self.position_embedding(positions)

class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


## 5. 檢查 patch shape


In [ ]:
patches = Patches(PATCH_SIZE)(x_train[:2])
print('patches shape:', patches.shape)
print('num patches:', NUM_PATCHES)


## 6. 建立小型 ViT 模型


In [ ]:
PROJECTION_DIM = 32
NUM_HEADS = 2
FF_DIM = 64

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
patches = Patches(PATCH_SIZE)(inputs)
encoded = PatchEncoder(NUM_PATCHES, PROJECTION_DIM)(patches)
x = TransformerEncoderBlock(PROJECTION_DIM, NUM_HEADS, FF_DIM)(encoded)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 7. 訓練與評估


In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_valid, y_valid), epochs=14, batch_size=32, verbose=0)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Mini ViT accuracy')
plt.ylim(0, 1.05)
plt.show()
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(model.evaluate(x_test, y_test, verbose=0, return_dict=True))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 8. 如何套用自己的圖片

將圖片 resize 到固定大小，確認 `IMAGE_SIZE` 可以被 `PATCH_SIZE` 整除。資料量不大時，ViT 可能需要更多 regularization，也建議和 CNN baseline 比較。


## 9. 小結

ViT 把圖片切成 patches，再把 patches 當成序列 token。理解這個轉換後，就能看懂 Transformer 如何延伸到影像分類。
